# Витрина треков

Одна строка — один трек. Витрина нужна для анализа популярности и длинного хвоста каталога.

## Что считаем

Прослушивания, слушателей, среднюю долю проигрывания, Listen+, рекомендации,
повторы, лайки и дизлайки.

In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE = PROJECT_ROOT / "data" / "yambda" / "flat" / "50m" / "multi_event.parquet"
MARTS = PROJECT_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 30)

from src.track_mart import build_track_mart, validate_all_marts

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
MART = MARTS / "mart_track.parquet"
assert STAGE_DB.exists(), "Сначала выполните ноутбук 01_source_quality.ipynb"
con = duckdb.connect()

## Сборка витрины

Новые столбцы и общая сверка находятся в `src/track_mart.py`.

In [2]:
build_track_mart(STAGE_DB, MART)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'rows': 934057}

## Состояние каталога

In [3]:
catalog = con.execute(f"""
SELECT
    count(*) AS tracks,
    count(*) FILTER (WHERE listens = 0) AS tracks_without_listens,
    count(*) FILTER (WHERE listens = 1) AS tracks_with_one_listen,
    median(listens) AS median_listens,
    quantile_cont(listens, 0.90) AS p90_listens,
    quantile_cont(listens, 0.99) AS p99_listens,
    max(listens) AS max_listens
FROM read_parquet('{MART.as_posix()}')
""").df()

catalog.rename(columns={
    "tracks": "треки", "tracks_without_listens": "без прослушиваний",
    "tracks_with_one_listen": "с одним прослушиванием",
    "median_listens": "медиана прослушиваний", "p90_listens": "90-й процентиль",
    "p99_listens": "99-й процентиль", "max_listens": "максимум прослушиваний",
})

,треки,без прослушиваний,с одним прослушиванием,медиана прослушиваний,90-й процентиль,99-й процентиль,максимум прослушиваний
0,934057,56889,312855,2.0,56.0,924.0,41984


## Концентрация прослушиваний

In [4]:
concentration = con.execute(f"""
WITH ranked AS (
    SELECT listens, ntile(100) OVER (ORDER BY listens DESC) AS popularity_group
    FROM read_parquet('{MART.as_posix()}')
)
SELECT
    round(sum(listens) FILTER (WHERE popularity_group = 1) * 100.0 / sum(listens), 2)
        AS top_1_pct_share,
    round(sum(listens) FILTER (WHERE popularity_group <= 10) * 100.0 / sum(listens), 2)
        AS top_10_pct_share
FROM ranked
""").df()

concentration.rename(columns={
    "top_1_pct_share": "доля прослушиваний у верхнего 1%, %",
    "top_10_pct_share": "доля прослушиваний у верхних 10%, %",
})

,"доля прослушиваний у верхнего 1%, %","доля прослушиваний у верхних 10%, %"
0,50.32,88.64


## Проверка

In [5]:
check = con.execute(f"""
SELECT count(*) = 934057 AS all_tracks,
       count(*) = count(DISTINCT item_id) AS unique_key,
       count(*) FILTER (WHERE listens = 0 AND listen_plus_rate IS NULL) = 56889 AS nulls_valid,
       min(listen_plus_rate) >= 0 AND max(listen_plus_rate) <= 1 AS rates_valid
FROM read_parquet('{MART.as_posix()}')
""").df()
assert check.all(axis=None), "Проверка витрины не пройдена"
print("Проверка пройдена: все треки сохранены, ключ уникален, доли корректны.")

Проверка пройдена: все треки сохранены, ключ уникален, доли корректны.


## Общая сверка трёх витрин

In [6]:
validate_all_marts(
    STAGE_DB,
    MARTS / "mart_product_day.parquet",
    MARTS / "mart_user.parquet",
    MARTS / "mart_track.parquet",
)

{'product_events_match_source': True,
 'product_listens_match_source': True,
 'user_key_is_unique': True,
 'track_key_is_unique': True}

## Вывод

Каталог сильно сконцентрирован: небольшая доля треков получает основную часть прослушиваний.
Сравнивать процентные показатели треков стоит только после минимального порога в 10–20 прослушиваний.